In [ ]:
import pandas as pd
import glob
import geopandas as gpd
from shapely.geometry import Point
import re

In [64]:
kaggle_files = glob.glob("./*.csv")
kaggle_dfs = [pd.read_csv(file) for file in kaggle_files]
df_kaggle = pd.concat(kaggle_dfs, ignore_index=True)

In [65]:
df_kaggle.tail()

,latitude,longitude,postal_code,address,closest_mrt,closest_mrt_dist,cbd_dist,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,year,years_remaining,remaining_lease
896641,1.338745,103.847253,311099.0,99B LOR 2 TOA PAYOH,Braddell MRT Station,197.208781,6180.426023,2014-09,TOA PAYOH,EXECUTIVE,99B,LOR 2 TOA PAYOH,10 TO 12,145.0,Apartment,1993,850000.0,NaN,78,NaN
896642,1.339016,103.847449,312099.0,99C LOR 2 TOA PAYOH,Braddell MRT Station,176.198624,6208.782458,2012-03,TOA PAYOH,EXECUTIVE,99C,LOR 2 TOA PAYOH,11 TO 15,149.0,Apartment,1993,862000.0,NaN,80,NaN
896643,1.339016,103.847449,312099.0,99C LOR 2 TOA PAYOH,Braddell MRT Station,176.198624,6208.782458,2012-06,TOA PAYOH,EXECUTIVE,99C,LOR 2 TOA PAYOH,04 TO 06,148.0,Apartment,1993,820000.0,NaN,80,NaN
896644,1.339016,103.847449,312099.0,99C LOR 2 TOA PAYOH,Braddell MRT Station,176.198624,6208.782458,2013-10,TOA PAYOH,EXECUTIVE,99C,LOR 2 TOA PAYOH,10 TO 12,148.0,Apartment,1993,905000.0,NaN,79,NaN
896645,1.339016,103.847449,312099.0,99C LOR 2 TOA PAYOH,Braddell MRT Station,176.198624,6208.782458,2013-12,TOA PAYOH,EXECUTIVE,99C,LOR 2 TOA PAYOH,07 TO 09,145.0,Apartment,1993,845000.0,NaN,79,NaN


In [66]:
df_kaggle.dtypes

latitude               float64
longitude              float64
postal_code            float64
address                 object
closest_mrt             object
closest_mrt_dist       float64
cbd_dist               float64
month                   object
town                    object
flat_type               object
block                   object
street_name             object
storey_range            object
floor_area_sqm         float64
flat_model              object
lease_commence_date      int64
resale_price           float64
year                   float64
years_remaining          int64
remaining_lease         object
dtype: object

In [ ]:
df_kaggle["postal_code"] = pd.to_numeric(df_kaggle["postal_code"], errors="coerce").astype("Int64")
df_kaggle["floor_area_sqm"] = pd.to_numeric(df_kaggle["floor_area_sqm"], errors="coerce").astype("int64")
df_kaggle["resale_price"] = pd.to_numeric(df_kaggle["resale_price"], errors="coerce").astype("int64")

# sort by month
df_kaggle = df_kaggle.sort_values(by=["month", "street_name"])
df_kaggle["month"] = pd.to_datetime(df_kaggle["month"], format="%Y-%m")

df_kaggle = df_kaggle.drop(columns=["year", "remaining_lease"], errors="ignore")

# year & month as separate columns
df_kaggle["year"] = df_kaggle["month"].dt.year
df_kaggle["month"] = df_kaggle["month"].dt.month

In [68]:
df_kaggle.head()

,latitude,longitude,postal_code,address,closest_mrt,closest_mrt_dist,cbd_dist,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,years_remaining,year
5495,1.327963,103.844827,320103,103 AH HOOD RD,Toa Payoh MRT Station,596.867091,5023.661065,1,KALLANG/WHAMPOA,5 ROOM,103,AH HOOD RD,16 TO 18,122,IMPROVED,1981,150000,90,1990
5496,1.327963,103.844827,320103,103 AH HOOD RD,Toa Payoh MRT Station,596.867091,5023.661065,1,KALLANG/WHAMPOA,5 ROOM,103,AH HOOD RD,10 TO 12,118,IMPROVED,1981,182000,90,1990
4292,1.320439,103.882420,380102,102 ALJUNIED CRES,Aljunied MRT Station,446.293460,5397.533295,1,GEYLANG,3 ROOM,102,ALJUNIED CRES,07 TO 09,67,NEW GENERATION,1978,57000,87,1990
8831,1.319055,103.882901,380106,106 ALJUNIED CRES,Aljunied MRT Station,289.965205,5316.286855,1,GEYLANG,3 ROOM,106,ALJUNIED CRES,01 TO 03,67,NEW GENERATION,1979,61300,88,1990
11186,1.319175,103.885322,380108,108 ALJUNIED CRES,Aljunied MRT Station,405.255803,5507.861454,1,GEYLANG,3 ROOM,108,ALJUNIED CRES,04 TO 06,68,NEW GENERATION,1981,68100,90,1990


In [69]:
# split 'storey_range' into lower & upper bounds
def split_storey_range(storey_range):
    try:
        lower, upper = storey_range.split(" TO ")
        return int(lower), int(upper)
    except ValueError:
        return None, None

df_kaggle[["storey_lower", "storey_upper"]] = df_kaggle["storey_range"].apply(
    lambda x: pd.Series(split_storey_range(x))
)

df_kaggle = df_kaggle.drop(columns=["storey_range"], errors="ignore")

##### Standardize cell content

In [ ]:
# standardize 'flat_type' to 'MULTI-GENERATION' with hyphen
df_kaggle['flat_type'] = df_kaggle['flat_type'].replace('MULTI GENERATION', 'MULTI-GENERATION')

# standardize 'flat_model' to all caps
df_kaggle['flat_model'] = df_kaggle['flat_model'].str.upper()

# rename 'town' column to 'planning_area'
df_kaggle = df_kaggle.rename(columns={"town": "planning_area"})

In [ ]:
df_kaggle = df_kaggle.rename(columns={"town": "planning_area"})

# change KALLANG/WHAMPOA to KALLANG
df_kaggle["planning_area"] = df_kaggle["planning_area"].replace("KALLANG/WHAMPOA", "KALLANG")

In [78]:
df_kaggle.columns

Index(['latitude', 'longitude', 'postal_code', 'address', 'closest_mrt',
       'closest_mrt_dist', 'cbd_dist', 'month', 'planning_area', 'flat_type',
       'block', 'street_name', 'floor_area_sqm', 'flat_model',
       'lease_commence_date', 'resale_price', 'years_remaining', 'year',
       'storey_lower', 'storey_upper'],
      dtype='object')

In [79]:
column_order = [
    "year", "month",
    "latitude", "longitude", "postal_code",
    "planning_area", "block", "street_name", 
    "flat_type", "flat_model", 
    "storey_lower", "storey_upper", "floor_area_sqm", 
    "lease_commence_date", "years_remaining",
    "closest_mrt", "closest_mrt_dist", "cbd_dist",
    "resale_price"
]

df_kaggle = df_kaggle[column_order]

In [80]:
df_kaggle

,year,month,latitude,longitude,postal_code,planning_area,block,street_name,flat_type,flat_model,storey_lower,storey_upper,floor_area_sqm,lease_commence_date,years_remaining,closest_mrt,closest_mrt_dist,cbd_dist,resale_price
5495,1990,1,1.327963,103.844827,320103,KALLANG/WHAMPOA,103,AH HOOD RD,5 ROOM,IMPROVED,16,18,122,1981,90,Toa Payoh MRT Station,596.867091,5023.661065,150000
5496,1990,1,1.327963,103.844827,320103,KALLANG/WHAMPOA,103,AH HOOD RD,5 ROOM,IMPROVED,10,12,118,1981,90,Toa Payoh MRT Station,596.867091,5023.661065,182000
4292,1990,1,1.320439,103.882420,380102,GEYLANG,102,ALJUNIED CRES,3 ROOM,NEW GENERATION,7,9,67,1978,87,Aljunied MRT Station,446.293460,5397.533295,57000
8831,1990,1,1.319055,103.882901,380106,GEYLANG,106,ALJUNIED CRES,3 ROOM,NEW GENERATION,1,3,67,1979,88,Aljunied MRT Station,289.965205,5316.286855,61300
11186,1990,1,1.319175,103.885322,380108,GEYLANG,108,ALJUNIED CRES,3 ROOM,NEW GENERATION,4,6,68,1981,90,Aljunied MRT Station,405.255803,5507.861454,68100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
834546,2023,4,1.415191,103.832902,760828,YISHUN,828,YISHUN ST 81,EXECUTIVE,APARTMENT,10,12,142,1988,64,Khatib MRT Station,242.585206,14759.708167,865000
835248,2023,4,1.415715,103.833410,760840,YISHUN,840,YISHUN ST 81,3 ROOM,MODEL A,4,6,73,1988,64,Khatib MRT Station,190.560783,14809.421378,412000
837710,2023,4,1.413545,103.836971,760872,YISHUN,872,YISHUN ST 81,5 ROOM,IMPROVED,1,3,127,1988,64,Khatib MRT Station,614.307170,14522.828800,640000
838053,2023,4,1.414442,103.836118,760879,YISHUN,879,YISHUN ST 81,4 ROOM,SIMPLIFIED,4,6,84,1987,63,Khatib MRT Station,477.228688,14632.029240,403000


##### Fix central area issue

In [103]:
# check planning areas
unique_planning_areas= df_kaggle['planning_area'].unique()

reference_planning_area = pd.read_csv("../resale_prices_cleaned/planning_areas.csv")
reference_planning_area_list = reference_planning_area['Planning Area'].tolist()

missing_towns = [town for town in unique_planning_areas if town not in reference_planning_area_list]

missing_towns, unique_planning_areas, reference_planning_area_list

(['CENTRAL AREA'],
 array(['KALLANG', 'GEYLANG', 'ANG MO KIO', 'CENTRAL AREA', 'BEDOK',
        'BUKIT MERAH', 'JURONG WEST', 'BUKIT BATOK', 'CLEMENTI',
        'QUEENSTOWN', 'BUKIT TIMAH', 'HOUGANG', 'TOA PAYOH', 'JURONG EAST',
        'SERANGOON', 'MARINE PARADE', 'WOODLANDS', 'SENGKANG', 'BISHAN',
        'TAMPINES', 'CHOA CHU KANG', 'YISHUN', 'LIM CHU KANG', 'SEMBAWANG',
        'BUKIT PANJANG', 'PASIR RIS', 'PUNGGOL'], dtype=object),
 ['BEDOK',
  'BUKIT TIMAH',
  'BUKIT BATOK',
  'BUKIT MERAH',
  'CENTRAL WATER CATCHMENT',
  'DOWNTOWN CORE',
  'CHANGI',
  'CHANGI BAY',
  'LIM CHU KANG',
  'BOON LAY',
  'WESTERN WATER CATCHMENT',
  'WOODLANDS',
  'MARINE PARADE',
  'NEWTON',
  'NORTH-EASTERN ISLANDS',
  'ORCHARD',
  'PASIR RIS',
  'PIONEER',
  'PUNGGOL',
  'QUEENSTOWN',
  'SEMBAWANG',
  'SIMPANG',
  'TAMPINES',
  'TANGLIN',
  'TUAS',
  'WESTERN ISLANDS',
  'SOUTHERN ISLANDS',
  'BUKIT PANJANG',
  'BISHAN',
  'ANG MO KIO',
  'GEYLANG',
  'STRAITS VIEW',
  'JURONG EAST',
  'HOUGANG',

In [84]:
central_area_df = df_kaggle[df_kaggle["planning_area"] == "CENTRAL AREA"]
central_area_df

,year,month,latitude,longitude,postal_code,planning_area,block,street_name,flat_type,flat_model,storey_lower,storey_upper,floor_area_sqm,lease_commence_date,years_remaining,closest_mrt,closest_mrt_dist,cbd_dist,resale_price
95745,1990,1,1.296853,103.853348,180232,CENTRAL AREA,232,BAIN ST,4 ROOM,IMPROVED,22,24,82,1980,89,Bras Basah MRT Station,270.530289,1548.662078,100000
242014,1990,1,1.306171,103.850106,210662,CENTRAL AREA,662,BUFFALO RD,4 ROOM,IMPROVED,13,15,85,1982,91,Little India MRT Station,125.291360,2565.567896,88000
242205,1990,1,1.305680,103.850858,210663,CENTRAL AREA,663,BUFFALO RD,4 ROOM,IMPROVED,16,18,82,1982,91,Little India MRT Station,206.080736,2508.355468,80000
242315,1990,1,1.306132,103.851113,210664,CENTRAL AREA,664,BUFFALO RD,3 ROOM,IMPROVED,1,3,60,1982,91,Little India MRT Station,184.098418,2557.953805,67000
262494,1990,1,1.287378,103.839407,162008,CENTRAL AREA,8,JLN KUKOH,2 ROOM,IMPROVED,10,12,53,1971,80,Chinatown MRT Station,727.479135,1409.321548,30000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810455,2023,3,1.306066,103.854502,200632,CENTRAL AREA,632,VEERASAMY RD,4 ROOM,MODEL A,13,15,102,1985,61,Jalan Besar MRT Station,132.651063,2575.289076,660000
811098,2023,3,1.306589,103.855219,200635,CENTRAL AREA,635,VEERASAMY RD,3 ROOM,MODEL A,1,3,72,1985,61,Jalan Besar MRT Station,156.931300,2644.504082,432000
739967,2023,3,1.298609,103.852267,180263,CENTRAL AREA,263,WATERLOO ST,3 ROOM,IMPROVED,19,21,60,1978,54,Bencoolen MRT Station,215.664994,1729.299527,525000
724047,2023,4,1.275987,103.841192,85601,CENTRAL AREA,1F,CANTONMENT RD,4 ROOM,TYPE S1,10,12,95,2011,87,Outram Park MRT Station,455.052615,1366.341045,1000000


In [105]:
central_area_df['street_name'].unique()

array(['BAIN ST', 'BUFFALO RD', 'JLN KUKOH', 'OUTRAM PK', 'QUEEN ST',
       'ROCHOR RD', 'ROWELL RD', 'SELEGIE RD', 'SHORT ST', 'SMITH ST',
       'UPP CROSS ST', 'VEERASAMY RD', 'WATERLOO ST', 'KELANTAN RD',
       'NEW MKT RD', 'TG PAGAR PLAZA', 'SAGO LANE', 'CHANDER RD',
       'JLN BERSEH', 'CHIN SWEE RD', 'KRETA AYER RD', 'OUTRAM HILL',
       'KLANG LANE', 'CANTONMENT RD'], dtype=object)

In [ ]:
# map rows with 'central area' to planning area using street name
street_to_planning_area_mapping = {
    'BAIN ST': 'ROCHOR',
    'BUFFALO RD': 'ROCHOR',
    'JLN KUKOH': 'OUTRAM',
    'OUTRAM PK': 'OUTRAM',
    'QUEEN ST': 'ROCHOR',
    'ROCHOR RD': 'ROCHOR',
    'ROWELL RD': 'ROCHOR',
    'SELEGIE RD': 'ROCHOR',
    'SHORT ST': 'ROCHOR',
    'SMITH ST': 'OUTRAM',
    'UPP CROSS ST': 'OUTRAM',
    'VEERASAMY RD': 'ROCHOR',
    'WATERLOO ST': 'ROCHOR',
    'KELANTAN RD': 'ROCHOR',
    'NEW MKT RD': 'OUTRAM',
    'TG PAGAR PLAZA': 'OUTRAM',
    'SAGO LANE': 'OUTRAM',
    'CHANDER RD': 'ROCHOR',
    'JLN BERSEH': 'ROCHOR',
    'CHIN SWEE RD': 'OUTRAM',
    'KRETA AYER RD': 'OUTRAM',
    'OUTRAM HILL': 'OUTRAM',
    'KLANG LANE': 'ROCHOR',
    'CANTONMENT RD': 'OUTRAM'
}

df_kaggle['planning_area'] = df_kaggle.apply(
    lambda x: street_to_planning_area_mapping[x['street_name']] if x['planning_area'] == 'CENTRAL AREA' else x['planning_area'],
    axis=1
)

In [107]:
df_kaggle['planning_area'].unique()

array(['KALLANG', 'GEYLANG', 'ANG MO KIO', 'ROCHOR', 'BEDOK',
       'BUKIT MERAH', 'JURONG WEST', 'BUKIT BATOK', 'CLEMENTI',
       'QUEENSTOWN', 'BUKIT TIMAH', 'HOUGANG', 'OUTRAM', 'TOA PAYOH',
       'JURONG EAST', 'SERANGOON', 'MARINE PARADE', 'WOODLANDS',
       'SENGKANG', 'BISHAN', 'TAMPINES', 'CHOA CHU KANG', 'YISHUN',
       'LIM CHU KANG', 'SEMBAWANG', 'BUKIT PANJANG', 'PASIR RIS',
       'PUNGGOL'], dtype=object)

##### Add electoral boundary & ruling party

In [ ]:
# add electoral boundary by lat,long
geojson_files = {
    (2006, 2010): "../electoral_boundary/ElectoralBoundary2006GEOJSON.geojson",
    (2011, 2014): "../electoral_boundary/ElectoralBoundary2011GEOJSON.geojson",
    (2015, 2019): "../electoral_boundary/ElectoralBoundary2015GEOJSON.geojson",
    (2020, 2023): "../electoral_boundary/ElectoralBoundary2020GEOJSON.geojson"
}

df_kaggle['geometry'] = [Point(xy) for xy in zip(df_kaggle['longitude'], df_kaggle['latitude'])]
gdf_kaggle = gpd.GeoDataFrame(df_kaggle, geometry=df_kaggle['geometry'], crs="EPSG:4326")

results = []

# extract ED_DESC from "Description" field for older GeoJSON files
def extract_ed_desc(description):
    match = re.search(r"<th>ED_DESC<\/th>\s*<td>(.*?)<\/td>", description)
    return match.group(1).strip() if match else None

for (start_year, end_year), geojson_path in geojson_files.items():
    gdf_boundaries = gpd.read_file(geojson_path)

    # ensure right ED_DESC column
    if "ED_DESC" not in gdf_boundaries.columns:
        if "Description" in gdf_boundaries.columns:
            gdf_boundaries["ED_DESC"] = gdf_boundaries["Description"].apply(extract_ed_desc)
        else:
            gdf_boundaries["ED_DESC"] = None 

    df_subset = gdf_kaggle[(gdf_kaggle['year'] >= start_year) & (gdf_kaggle['year'] <= end_year)]

    if not df_subset.empty:
        # spatial join
        gdf_result = gpd.sjoin(df_subset, gdf_boundaries, how="left", predicate="within")
        gdf_result["electoral_boundary"] = gdf_result["ED_DESC"]
        results.append(gdf_result)

final_gdf_kaggle = pd.concat(results, ignore_index=True)

# flats before 2006
df_pre_2006 = gdf_kaggle[gdf_kaggle['year'] < 2006].copy()
df_pre_2006['electoral_boundary'] = None 

final_gdf_kaggle = pd.concat([df_pre_2006, final_gdf_kaggle], ignore_index=True)

final_gdf_kaggle = final_gdf_kaggle[[
                        "year", "month",
                        "latitude", "longitude", "postal_code",
                        "planning_area", "block", "street_name", 
                        "flat_type", "flat_model", 
                        "storey_lower", "storey_upper", "floor_area_sqm", 
                        "lease_commence_date", "years_remaining",
                        "closest_mrt", "closest_mrt_dist", "cbd_dist",
                        'electoral_boundary',
                        "resale_price"
                    ]]

final_gdf_kaggle

,year,month,latitude,longitude,postal_code,planning_area,block,street_name,flat_type,flat_model,storey_lower,storey_upper,floor_area_sqm,lease_commence_date,years_remaining,closest_mrt,closest_mrt_dist,cbd_dist,electoral_boundary,resale_price
0,1990,1,1.327963,103.844827,320103,KALLANG,103,AH HOOD RD,5 ROOM,IMPROVED,16,18,122,1981,90,Toa Payoh MRT Station,596.867091,5023.661065,None,150000
1,1990,1,1.327963,103.844827,320103,KALLANG,103,AH HOOD RD,5 ROOM,IMPROVED,10,12,118,1981,90,Toa Payoh MRT Station,596.867091,5023.661065,None,182000
2,1990,1,1.320439,103.882420,380102,GEYLANG,102,ALJUNIED CRES,3 ROOM,NEW GENERATION,7,9,67,1978,87,Aljunied MRT Station,446.293460,5397.533295,None,57000
3,1990,1,1.319055,103.882901,380106,GEYLANG,106,ALJUNIED CRES,3 ROOM,NEW GENERATION,1,3,67,1979,88,Aljunied MRT Station,289.965205,5316.286855,None,61300
4,1990,1,1.319175,103.885322,380108,GEYLANG,108,ALJUNIED CRES,3 ROOM,NEW GENERATION,4,6,68,1981,90,Aljunied MRT Station,405.255803,5507.861454,None,68100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
896641,2023,4,1.415191,103.832902,760828,YISHUN,828,YISHUN ST 81,EXECUTIVE,APARTMENT,10,12,142,1988,64,Khatib MRT Station,242.585206,14759.708167,NEE SOON,865000
896642,2023,4,1.415715,103.833410,760840,YISHUN,840,YISHUN ST 81,3 ROOM,MODEL A,4,6,73,1988,64,Khatib MRT Station,190.560783,14809.421378,NEE SOON,412000
896643,2023,4,1.413545,103.836971,760872,YISHUN,872,YISHUN ST 81,5 ROOM,IMPROVED,1,3,127,1988,64,Khatib MRT Station,614.307170,14522.828800,NEE SOON,640000
896644,2023,4,1.414442,103.836118,760879,YISHUN,879,YISHUN ST 81,4 ROOM,SIMPLIFIED,4,6,84,1987,63,Khatib MRT Station,477.228688,14632.029240,NEE SOON,403000


In [128]:
# add ruling party by rules
def get_ruling_party(row):
    # no electoral boundary
    if pd.isna(row['electoral_boundary']):
        return None
    
    eb = row['electoral_boundary']
    year = row['year']
    
    if 2006 <= year <= 2010:
        return "WP" if eb == "HOUGANG" else "SDA" if eb == "POTONG PASIR" else "PAP"
    elif 2011 <= year <= 2012:
        return "WP" if eb in ["ALJUNIED", "HOUGANG"] else "PAP"
    elif 2013 <= year <= 2014:
        return "WP" if eb in ["ALJUNIED", "HOUGANG", "PUNGGOL EAST"] else "PAP"
    elif 2015 <= year <= 2019:
        return "WP" if eb in ["ALJUNIED", "HOUGANG"] else "PAP"
    elif 2020 <= year <= 2023:
        return "WP" if eb in ["ALJUNIED", "HOUGANG", "SENGKANG"] else "PAP"
    
final_gdf_kaggle["ruling_party"] = final_gdf_kaggle.apply(get_ruling_party, axis=1)

final_gdf_kaggle = final_gdf_kaggle[[
                        "year", "month",
                        "latitude", "longitude", "postal_code",
                        "planning_area", "block", "street_name", 
                        "flat_type", "flat_model", 
                        "storey_lower", "storey_upper", "floor_area_sqm", 
                        "lease_commence_date", "years_remaining",
                        "closest_mrt", "closest_mrt_dist", "cbd_dist",
                        'electoral_boundary', "ruling_party",
                        "resale_price"
                    ]]
final_gdf_kaggle

,year,month,latitude,longitude,postal_code,planning_area,block,street_name,flat_type,flat_model,...,storey_upper,floor_area_sqm,lease_commence_date,years_remaining,closest_mrt,closest_mrt_dist,cbd_dist,electoral_boundary,ruling_party,resale_price
0,1990,1,1.327963,103.844827,320103,KALLANG,103,AH HOOD RD,5 ROOM,IMPROVED,...,18,122,1981,90,Toa Payoh MRT Station,596.867091,5023.661065,None,None,150000
1,1990,1,1.327963,103.844827,320103,KALLANG,103,AH HOOD RD,5 ROOM,IMPROVED,...,12,118,1981,90,Toa Payoh MRT Station,596.867091,5023.661065,None,None,182000
2,1990,1,1.320439,103.882420,380102,GEYLANG,102,ALJUNIED CRES,3 ROOM,NEW GENERATION,...,9,67,1978,87,Aljunied MRT Station,446.293460,5397.533295,None,None,57000
3,1990,1,1.319055,103.882901,380106,GEYLANG,106,ALJUNIED CRES,3 ROOM,NEW GENERATION,...,3,67,1979,88,Aljunied MRT Station,289.965205,5316.286855,None,None,61300
4,1990,1,1.319175,103.885322,380108,GEYLANG,108,ALJUNIED CRES,3 ROOM,NEW GENERATION,...,6,68,1981,90,Aljunied MRT Station,405.255803,5507.861454,None,None,68100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
896641,2023,4,1.415191,103.832902,760828,YISHUN,828,YISHUN ST 81,EXECUTIVE,APARTMENT,...,12,142,1988,64,Khatib MRT Station,242.585206,14759.708167,NEE SOON,PAP,865000
896642,2023,4,1.415715,103.833410,760840,YISHUN,840,YISHUN ST 81,3 ROOM,MODEL A,...,6,73,1988,64,Khatib MRT Station,190.560783,14809.421378,NEE SOON,PAP,412000
896643,2023,4,1.413545,103.836971,760872,YISHUN,872,YISHUN ST 81,5 ROOM,IMPROVED,...,3,127,1988,64,Khatib MRT Station,614.307170,14522.828800,NEE SOON,PAP,640000
896644,2023,4,1.414442,103.836118,760879,YISHUN,879,YISHUN ST 81,4 ROOM,SIMPLIFIED,...,6,84,1987,63,Khatib MRT Station,477.228688,14632.029240,NEE SOON,PAP,403000


#### Add region

In [140]:
region_map = {
    "NORTH": ["WOODLANDS", "YISHUN", "SEMBAWANG", "LIM CHU KANG"],
    "NORTH-EAST": ["ANG MO KIO", "HOUGANG", "SENGKANG", "PUNGGOL", "SERANGOON"],
    "EAST": ["BEDOK", "TAMPINES", "PASIR RIS"],
    "WEST": ["JURONG EAST", "JURONG WEST", "BUKIT BATOK", "CHOA CHU KANG", "BUKIT PANJANG", "CLEMENTI"],
    "CENTRAL": ["BISHAN", "KALLANG", "TOA PAYOH", "ROCHOR", "BUKIT TIMAH", "OUTRAM",
                "GEYLANG", "MARINE PARADE", "QUEENSTOWN", "BUKIT MERAH"]
}

final_gdf_kaggle["region"] = final_gdf_kaggle["planning_area"].map({town: region for region, towns in region_map.items() for town in towns})

final_gdf_kaggle = final_gdf_kaggle[[
                        "year", "month",
                        "latitude", "longitude", "postal_code",
                        "region", "planning_area", "block", "street_name", 
                        "flat_type", "flat_model", 
                        "storey_lower", "storey_upper", "floor_area_sqm", 
                        "lease_commence_date", "years_remaining",
                        "closest_mrt", "closest_mrt_dist", "cbd_dist",
                        'electoral_boundary', "ruling_party",
                        "resale_price"
                    ]]
final_gdf_kaggle

,year,month,latitude,longitude,postal_code,region,planning_area,block,street_name,flat_type,...,storey_upper,floor_area_sqm,lease_commence_date,years_remaining,closest_mrt,closest_mrt_dist,cbd_dist,electoral_boundary,ruling_party,resale_price
0,1990,1,1.327963,103.844827,320103,CENTRAL,KALLANG,103,AH HOOD RD,5 ROOM,...,18,122,1981,90,Toa Payoh MRT Station,596.867091,5023.661065,None,None,150000
1,1990,1,1.327963,103.844827,320103,CENTRAL,KALLANG,103,AH HOOD RD,5 ROOM,...,12,118,1981,90,Toa Payoh MRT Station,596.867091,5023.661065,None,None,182000
2,1990,1,1.320439,103.882420,380102,CENTRAL,GEYLANG,102,ALJUNIED CRES,3 ROOM,...,9,67,1978,87,Aljunied MRT Station,446.293460,5397.533295,None,None,57000
3,1990,1,1.319055,103.882901,380106,CENTRAL,GEYLANG,106,ALJUNIED CRES,3 ROOM,...,3,67,1979,88,Aljunied MRT Station,289.965205,5316.286855,None,None,61300
4,1990,1,1.319175,103.885322,380108,CENTRAL,GEYLANG,108,ALJUNIED CRES,3 ROOM,...,6,68,1981,90,Aljunied MRT Station,405.255803,5507.861454,None,None,68100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
896641,2023,4,1.415191,103.832902,760828,NORTH,YISHUN,828,YISHUN ST 81,EXECUTIVE,...,12,142,1988,64,Khatib MRT Station,242.585206,14759.708167,NEE SOON,PAP,865000
896642,2023,4,1.415715,103.833410,760840,NORTH,YISHUN,840,YISHUN ST 81,3 ROOM,...,6,73,1988,64,Khatib MRT Station,190.560783,14809.421378,NEE SOON,PAP,412000
896643,2023,4,1.413545,103.836971,760872,NORTH,YISHUN,872,YISHUN ST 81,5 ROOM,...,3,127,1988,64,Khatib MRT Station,614.307170,14522.828800,NEE SOON,PAP,640000
896644,2023,4,1.414442,103.836118,760879,NORTH,YISHUN,879,YISHUN ST 81,4 ROOM,...,6,84,1987,63,Khatib MRT Station,477.228688,14632.029240,NEE SOON,PAP,403000


In [141]:
final_gdf_kaggle.to_csv("cleaned_hdb_resale_data.csv", index=False)

## (A) Town Level

In [131]:
df_town_level_year_flat_type = final_gdf_kaggle.groupby(["year", "planning_area", "flat_type"]).agg(
    avg_resale_price=("resale_price", "mean"),
    median_resale_price=("resale_price", "median"),
    total_transactions=("resale_price", "count"),
    avg_floor_area=("floor_area_sqm", "mean"),
    avg_remaining_lease=("years_remaining", "mean"),
    avg_mrt_distance=("closest_mrt_dist", "mean"),
    avg_cbd_distance=("cbd_dist", "mean")
).reset_index()

df_town_level_year_flat_type

,year,planning_area,flat_type,avg_resale_price,median_resale_price,total_transactions,avg_floor_area,avg_remaining_lease,avg_mrt_distance,avg_cbd_distance
0,1990,ANG MO KIO,1 ROOM,7.770833e+03,8000.0,24,31.000000,86.000000,NaN,NaN
1,1990,ANG MO KIO,2 ROOM,2.510833e+04,23000.0,12,45.000000,95.000000,357.423622,10257.700017
2,1990,ANG MO KIO,3 ROOM,4.644389e+04,47000.0,1096,71.499088,88.287409,639.520721,9735.328305
3,1990,ANG MO KIO,4 ROOM,7.706832e+04,75000.0,368,92.942935,88.103261,602.672782,9739.138430
4,1990,ANG MO KIO,5 ROOM,1.307095e+05,130000.0,128,120.351562,88.585938,694.200289,9743.971289
...,...,...,...,...,...,...,...,...,...,...
3944,2023,YISHUN,3 ROOM,3.826468e+05,378000.0,153,68.764706,70.163399,774.735620,16042.123346
3945,2023,YISHUN,4 ROOM,4.843981e+05,479500.0,252,93.690476,74.039683,956.959060,15785.664166
3946,2023,YISHUN,5 ROOM,6.371329e+05,635000.0,95,117.400000,79.284211,940.729756,15740.229401
3947,2023,YISHUN,EXECUTIVE,8.194667e+05,800000.0,15,148.800000,64.066667,883.643875,15867.634211


In [132]:
df_town_level_year_flat_type.to_csv("resale_data_year_planning_area_flat_type.csv", index=False)

In [135]:
df_town_level_year = final_gdf_kaggle.groupby(["year", "planning_area"]).agg(
    avg_resale_price=("resale_price", "mean"),
    median_resale_price=("resale_price", "median"),
    total_transactions=("resale_price", "count"),
    avg_floor_area=("floor_area_sqm", "mean"),
    avg_remaining_lease=("years_remaining", "mean"),
    avg_mrt_distance=("closest_mrt_dist", "mean"),
    avg_cbd_distance=("cbd_dist", "mean")
).reset_index()

df_town_level_year

,year,planning_area,avg_resale_price,median_resale_price,total_transactions,avg_floor_area,avg_remaining_lease,avg_mrt_distance,avg_cbd_distance
0,1990,ANG MO KIO,59264.292383,47200.0,1628,79.394963,88.285012,633.166924,9740.990746
1,1990,BEDOK,70226.199678,57000.0,1242,86.627214,88.518519,808.656018,10189.943945
2,1990,BISHAN,92737.735849,65500.0,106,89.047170,91.358491,413.762159,7820.168766
3,1990,BUKIT BATOK,87756.431611,71400.0,658,98.648936,93.653495,590.111920,13609.767856
4,1990,BUKIT MERAH,65934.881690,50000.0,710,73.200000,84.976056,695.492139,3442.449856
...,...,...,...,...,...,...,...,...,...
901,2023,SERANGOON,599196.768519,577500.0,108,99.000000,64.203704,1044.272397,8901.628985
902,2023,TAMPINES,576160.455487,565000.0,483,101.207039,68.672878,736.746165,13217.821338
903,2023,TOA PAYOH,593232.205128,510000.0,195,85.861538,63.256410,530.380552,6075.399559
904,2023,WOODLANDS,524988.255892,520000.0,594,102.033670,77.865320,602.072253,18376.758909


In [136]:
df_town_level_year.to_csv("resale_data_year_planning_area.csv", index=False)

## (B) Regional Level

In [142]:
df_region_level_year = final_gdf_kaggle.groupby(["year", "region"]).agg(
    avg_resale_price=("resale_price", "mean"),
    median_resale_price=("resale_price", "median"),
    total_transactions=("resale_price", "count"),
    avg_floor_area=("floor_area_sqm", "mean"),
    avg_mrt_distance=("closest_mrt_dist", "mean"),
    avg_cbd_distance=("cbd_dist", "mean")
).reset_index()

df_region_level_year

,year,region,avg_resale_price,median_resale_price,total_transactions,avg_floor_area,avg_mrt_distance,avg_cbd_distance
0,1990,CENTRAL,61607.181266,45000.0,3950,74.184051,613.531763,5473.417698
1,1990,EAST,76622.552783,65800.0,2084,91.490403,727.690556,11343.788667
2,1990,NORTH,59326.524807,52400.0,907,87.674752,734.091560,17496.958543
3,1990,NORTH-EAST,70223.733568,52330.0,2556,84.481221,710.082559,9587.548674
4,1990,WEST,71504.794548,56000.0,3008,89.199468,777.348251,13882.355104
...,...,...,...,...,...,...,...,...
165,2023,CENTRAL,627277.863929,613500.0,1242,85.527375,560.932137,5322.894424
166,2023,EAST,560510.180077,550000.0,1044,99.903257,795.993075,12306.327977
167,2023,NORTH,510852.266715,500000.0,1376,96.218023,713.395968,17406.796046
168,2023,NORTH-EAST,564778.250556,560000.0,1800,94.161667,1086.064624,12189.098554


In [143]:
df_region_level_year.to_csv("resale_data_year_region.csv", index=False)